# Resonance Shift from Artificial Edge Delays in Baseband Simulation

In a sample-mode circuit simulation, every graph edge introduces one artificial
time-step delay $\Delta t$.  For a ring resonator whose closure passes through
$n$ such edges, this perturbs the resonant wavelength.

A naive analysis gives
$$|\Delta\lambda| = \lambda_\text{res}\,\frac{n\,\Delta t}{\tau_\text{rt}}$$
but this predicts shifts of tens of nanometres that are never observed in
practice.  The correct formula is
$$\boxed{|\Delta\lambda| = |\lambda_\text{res} - \lambda_c|\,\frac{n\,\Delta t}{\tau_\text{rt}}}$$
where $\lambda_c$ is the **carrier centre wavelength** used when fitting the
state-space models.  This notebook derives the formula from first principles
and verifies it numerically.

## 1. The Ring Resonance Condition

A ring resonator resonates when the round-trip optical phase equals an integer
multiple of $2\pi$:

$$\varphi_\text{rt}(f) = 2\pi\,\frac{n_g\,L_\text{rt}}{c}\,f = 2\pi m
\qquad m \in \mathbb{Z}$$

giving resonant frequencies
$$f_m = \frac{m\,c}{n_g\,L_\text{rt}}$$
and free spectral range $\text{FSR} = c / (n_g L_\text{rt})$.

## 2. Baseband Representation

The simulator works with the **complex envelope** $A(t)$ defined by
$$E(t) = \operatorname{Re}\!\left[A(t)\,e^{j2\pi f_c t}\right]$$
where $f_c = c/\lambda_c$ is a fixed carrier frequency.  Every component is
represented by a discrete-time state-space model fitted — via vector fitting —
to the physical transfer function at offset frequencies
$\delta\!f = f - f_c$ over a finite spectral band.

For a waveguide with group delay $\tau_\text{wg}$ the physical transfer
function is
$$H_\text{wg}(f) = e^{j2\pi f\,\tau_\text{wg}}$$
which at offset $\delta\!f = f - f_c$ becomes
$$H_\text{wg}(f_c + \delta\!f)
  = \underbrace{e^{j2\pi f_c\tau_\text{wg}}}_{\text{carrier phase}}
    \cdot\,e^{j2\pi\,\delta\!f\,\tau_\text{wg}}$$

The **carrier phase** $e^{j2\pi f_c\tau_\text{wg}}$ is a constant and is
captured by the poles and residues of the state-space model during fitting.
It contributes to the round-trip resonance condition at $\delta\!f = 0$ just
as in the physical ring.

## 3. Effect of an Artificial Edge Delay

In the discrete-time simulation each graph edge carries the previous time
step's output to the next component's input, introducing a delay of $\Delta t$.
This is a delay of the **complex envelope** only:

$$A(t) \longrightarrow A(t - \Delta t)$$

In the frequency domain the envelope delay multiplies by
$$H_\text{delay}(\delta\!f) = e^{-j2\pi\,\delta\!f\,\Delta t}$$

Crucially, there is **no separate carrier-phase term** $e^{-j2\pi f_c \Delta t}$.
That would appear if we were delaying the real optical field, but here we are
delaying the envelope only.  The carrier phase is already baked into the
state-space model.

### Round-trip resonance with $n$ artificial delays

Let $\varphi_\text{SS}(\delta\!f)$ be the round-trip phase contributed by
the state-space models.  Because the models capture the full physical
transfer function (including carrier phase), at offset $\delta\!f$:

$$\varphi_\text{SS}(\delta\!f) \approx
  \underbrace{2\pi f_c\,\tau_\text{rt}}_{\varphi_\text{carrier}}
  + 2\pi\,\delta\!f\,\tau_\text{rt}$$

Without artificial delays the resonance at $\delta\!f_0$ satisfies
$$\varphi_\text{carrier} + 2\pi\,\delta\!f_0\,\tau_\text{rt} = 2\pi m$$

With $n$ artificial delays of $\Delta t$ each, the total round-trip phase
picks up an extra $-2\pi\,\delta\!f\cdot n\Delta t$ from the envelope delays:

$$\varphi_\text{carrier}
  + 2\pi\,\delta\!f_\text{new}\,\tau_\text{rt}
  - 2\pi\,\delta\!f_\text{new}\cdot n\Delta t = 2\pi m$$

Subtracting the unperturbed condition:
$$2\pi(\delta\!f_\text{new} - \delta\!f_0)\,\tau_\text{rt}
  = 2\pi\,\delta\!f_\text{new}\cdot n\Delta t$$

$$\Delta(\delta\!f) = \delta\!f_\text{new} - \delta\!f_0
  \approx \delta\!f_0\,\frac{n\Delta t}{\tau_\text{rt} - n\Delta t}
  \approx \delta\!f_0\,\frac{n\Delta t}{\tau_\text{rt}}$$

Converting to wavelength using $\Delta\lambda \approx -\frac{\lambda_c^2}{c}\Delta(\delta\!f)$
and $\delta\!f_0 \approx -\frac{c}{\lambda_c^2}(\lambda_\text{res} - \lambda_c)$:

$$\boxed{|\Delta\lambda| = |\lambda_\text{res} - \lambda_c|\;\frac{n\,\Delta t}{\tau_\text{rt}}}$$

The shift is proportional to the **offset of the resonance from the carrier
centre wavelength**, not to $\lambda_\text{res}$ itself.  A resonance sitting
exactly at $\lambda_c$ experiences **zero shift** regardless of $n$ or $\Delta t$.

## 4. Numerical Verification

We build a single all-pass ring resonator with a transparent phase modulator
in the cavity.  The modulator (a non-SAX component) prevents the ring from
being pre-solved by the state-space group, so the resonance genuinely emerges
from the time-domain feedback — which is exactly the regime where artificial
delays matter.

We then:
1. Run an S-parameter simulation to locate the exact resonant wavelengths.
2. Run a sample-mode simulation and locate the resonance dips in the
   steady-state spectrum.
3. Measure the shift for each resonance and compare with the formula.

In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

from simphony.circuit.circuit import Circuit
from simphony.simulation.s_parameter import (
    SParameterSimulation, SParameterSimulationParameters)
from simphony.simulation.sample_mode import (
    SampleModeSimulation, SampleModeSimulationParameters)
from simphony.libraries.ideal.sources import OpticalCombSource, VoltageSource
from simphony.libraries.ideal.modulators import OpticalModulator
from simphony.libraries.siepic import half_ring, waveguide
from scipy.constants import speed_of_light

In [ ]:
# ── Ring parameters ───────────────────────────────────────────────────────────
HR       = dict(pol='te', gap=100, radius=5, width=500, thickness=220, coupling_length=0)
WG       = dict(pol='te', length=5.0, width=500, height=220, loss=100.0)
MOD      = dict(phase_coefficients=np.zeros(4), absorption_coefficients=np.zeros(4), length=1.0)
VS       = dict(steady_state_voltage=0.0)

# Physical ring parameters for the formula
ng       = 3.5
r        = HR['radius'] * 1e-6          # m
L_close  = WG['length'] * 1e-6          # m
L_rt     = np.pi * r + L_close          # actual ring path (half-arc + straight)
tau_rt   = ng * L_rt / speed_of_light   # round-trip group delay (s)
FSR      = (1.55e-6)**2 / (ng * L_rt)   # free spectral range (m)

DT       = 1e-14    # 10 fs sample period
N_STEPS  = 2000
TRANSIENT = 300

# Number of edges in the ring feedback loop:
#   half_ring -> mod  (1)
#   mod -> wg         (1)
#   wg  -> half_ring  (1)
N_EDGES  = 3

print(f"Ring path length  L_rt  = {L_rt*1e6:.3f} µm")
print(f"Round-trip time   τ_rt  = {tau_rt*1e15:.1f} fs")
print(f"Free spectral range FSR = {FSR*1e9:.2f} nm")
print(f"Artificial delays / round trip: {N_EDGES}")
print(f"n·Δt / τ_rt = {N_EDGES*DT/tau_rt:.4f}  ({N_EDGES*DT/tau_rt*100:.2f}%)")

In [ ]:
# ── S-parameter circuit (ring closed via modulator for fair comparison) ───────
sp_netlist = {
    'instances':   {'hr':'half_ring','mod':'modulator','wg':'waveguide','vs':'voltage_source'},
    'connections': {'hr,port_2':'mod,o0','mod,o1':'wg,o0','wg,o1':'hr,port_4','vs,e0':'mod,e0'},
    'ports':       {'in':'hr,port_1','out':'hr,port_3'},
}
sp_models  = {'half_ring':half_ring,'waveguide':waveguide,
              'modulator':OpticalModulator,'voltage_source':VoltageSource}
sp_circuit = Circuit(sp_netlist, sp_models)
sp_settings = {'hr':HR,'wg':WG,'mod':MOD,'vs':VS}

# ── Sample-mode circuit (source drives the ring) ──────────────────────────────
sm_netlist = {
    'instances':   {'hr':'half_ring','mod':'modulator','wg':'waveguide',
                    'vs':'voltage_source','source':'comb_source'},
    'connections': {'hr,port_2':'mod,o0','mod,o1':'wg,o0','wg,o1':'hr,port_4',
                    'vs,e0':'mod,e0','source,o0':'hr,port_1'},
    'ports':       {'out':'hr,port_3'},
}
sm_models  = {**sp_models, 'comb_source':OpticalCombSource}
sm_circuit = Circuit(sm_netlist, sm_models)

In [ ]:
# ── S-parameter ground truth ──────────────────────────────────────────────────
wl_sp = np.linspace(1.50, 1.60, 2001)   # µm
sp_sim = SParameterSimulation(sp_circuit, sp_settings, SParameterSimulationParameters())
sp_res = sp_sim.run(wl=wl_sp)
sp_T   = np.abs(np.array(sp_res.s_parameters[('in','out')]))**2

# Find resonance dip positions (minima in transmission)
dip_idx, _ = find_peaks(-sp_T, prominence=0.05)
sp_resonances = wl_sp[dip_idx] * 1e-6   # metres
print("S-parameter resonances (µm):", [f"{r*1e6:.4f}" for r in sp_resonances])

plt.figure(figsize=(10, 3))
plt.plot(wl_sp, sp_T, 'b-', lw=1.5)
plt.scatter(wl_sp[dip_idx], sp_T[dip_idx], color='red', zorder=5, s=60, label='Resonances')
plt.xlabel('Wavelength (µm)'); plt.ylabel('Transmission')
plt.title('All-Pass Ring: S-Parameter Transmission')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ── Sample-mode simulation ────────────────────────────────────────────────────
# Dense wavelength grid so we can resolve resonance positions accurately
lambda_c = 1.55e-6   # VF centre wavelength
sm_wl    = jnp.linspace(1.50, 1.60, 501) * 1e-6

VF = dict(model_order=None, min_model_order=2, max_model_order=30,
          num_frequency_samples=600, center_wavelength=lambda_c,
          spectral_range=(1.50e-6, 1.60e-6))

sm_settings = {
    'hr':     {'sax_settings': HR, 'vector_fitting_parameters': VF},
    'wg':     {'sax_settings': WG, 'vector_fitting_parameters': VF},
    'mod':    MOD,
    'vs':     VS,
    'source': {'wavelength': sm_wl, 'linewidth': 0.0},
}
sm_params = SampleModeSimulationParameters(
    optical_baseband_wavelengths=sm_wl, dt=DT, num_time_steps=N_STEPS)
sm_sim = SampleModeSimulation(sm_circuit, sm_settings,
                              tracked_ports={'out':'hr,port_3'},
                              simulation_parameters=sm_params)

print(f'Running sample-mode ({N_STEPS} steps × {len(sm_wl)} wavelengths, '
      f'λ_c = {lambda_c*1e9:.0f} nm) …')
sm_result = sm_sim.run(use_jit=True)
print('Done.')

In [ ]:
# ── Extract steady-state spectrum and find resonance dips ─────────────────────
out_amp  = np.array(sm_result.output_signals['out'].amplitude)  # (N, L, M)
sm_T     = np.mean(np.abs(out_amp[TRANSIENT:, :, 0])**2, axis=0)  # (L,)
sm_wl_nm = np.array(sm_wl) * 1e6   # µm for plotting

sm_dip_idx, _ = find_peaks(-sm_T, prominence=0.05)
sm_resonances  = np.array(sm_wl)[sm_dip_idx]   # metres
print('Sample-mode resonances (µm):', [f'{r*1e6:.4f}' for r in sm_resonances])

In [ ]:
# ── Match resonances and measure shifts ───────────────────────────────────────
print(f'\nCarrier centre wavelength  λ_c = {lambda_c*1e9:.2f} nm')
print(f'Artificial delays per ring feedback loop: {N_EDGES}')
print(f'Δt = {DT*1e15:.0f} fs,  τ_rt = {tau_rt*1e15:.1f} fs\n')

header = (f'{"λ_res (nm)":>12}  {"δλ = λ_res−λ_c (nm)":>22}  '
          f'{"Observed Δλ (nm)":>18}  {"Formula Δλ (nm)":>17}  {"Error":>8}')
print(header)
print('-' * len(header))

shifts_obs, shifts_formula, offsets = [], [], []

for sp_r in sp_resonances:
    # Find the closest sample-mode resonance
    if len(sm_resonances) == 0:
        continue
    closest_idx = np.argmin(np.abs(sm_resonances - sp_r))
    sm_r = sm_resonances[closest_idx]

    obs      = abs(sm_r - sp_r)              # observed shift (m)
    delta_lam = abs(sp_r - lambda_c)         # |λ_res − λ_c| (m)
    formula  = delta_lam * N_EDGES * DT / tau_rt   # predicted shift (m)
    error_pct = (obs - formula) / formula * 100 if formula > 0 else float('nan')

    shifts_obs.append(obs * 1e9)
    shifts_formula.append(formula * 1e9)
    offsets.append(delta_lam * 1e9)

    print(f'  {sp_r*1e9:>10.3f}  {(sp_r-lambda_c)*1e9:>+21.3f}  '
          f'{obs*1e9:>18.4f}  {formula*1e9:>17.4f}  {error_pct:>7.1f}%')

In [ ]:
# ── Visualisation: spectrum overlay + formula vs observed shift ───────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: transmission comparison
ax = axes[0]
ax.plot(wl_sp, sp_T, 'b-', lw=2, label='S-parameter (exact)', zorder=3)
ax.plot(sm_wl_nm, sm_T, 'r-', lw=1.5, alpha=0.85, label=f'Sample-mode  (λ_c={lambda_c*1e9:.0f} nm)')
ax.axvline(lambda_c * 1e6, color='k', ls=':', lw=1, label='λ_c')
ax.set_xlabel('Wavelength (µm)'); ax.set_ylabel('Transmission')
ax.set_title('Transmission: resonance dips shift away from λ_c')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Right: formula vs observed shift
ax = axes[1]
delta_lam_range = np.linspace(0, max(offsets) * 1.1, 200)
ax.plot(delta_lam_range,
        delta_lam_range * N_EDGES * DT / tau_rt * 1e9,
        'b-', lw=2, label=r'Formula: $|\delta\lambda|\cdot n\Delta t/\tau_\mathrm{rt}$')
ax.scatter(offsets, shifts_obs, s=80, color='red', zorder=5,
           label='Observed shift (simulation)')
ax.set_xlabel(r'$|\lambda_\mathrm{res} - \lambda_c|$  (nm)')
ax.set_ylabel('Resonance shift  (nm)')
ax.set_title('Shift scales linearly with distance from λ_c')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('baseband_delay_shift.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved baseband_delay_shift.png')

## 5. Why the Naive Formula Is Wrong by a Factor of $\lambda_\text{res}/\delta\lambda$

The naive derivation treats the artificial delay as if it were delaying the
**real optical field**, adding a carrier phase of $2\pi f_c\,\Delta t$ per
step.  This gives
$$|\Delta\lambda|_\text{naive} = \lambda_\text{res}\,\frac{n\,\Delta t}{\tau_\text{rt}}$$

The correct analysis recognises that the delay acts on the **complex envelope**,
adding only the offset phase $2\pi\,\delta\!f\,\Delta t$.  The ratio between
the two predictions is
$$\frac{|\Delta\lambda|_\text{naive}}{|\Delta\lambda|_\text{correct}}
  = \frac{\lambda_\text{res}}{|\lambda_\text{res} - \lambda_c|}$$

For a resonance 10 nm from the carrier centre at 1550 nm this ratio is
$1550/10 = 155$, explaining why the naive formula over-predicts by two orders
of magnitude.

In [ ]:
print('Comparison of the two formulae at each resonance:\n')
print(f'{"λ_res (nm)":>12}  {"δλ (nm)":>10}  '
      f'{"Correct (nm)":>14}  {"Naive (nm)":>12}  {"Ratio":>8}')
print('-' * 65)
for sp_r in sp_resonances:
    delta_lam = abs(sp_r - lambda_c)
    correct  = delta_lam * N_EDGES * DT / tau_rt
    naive    = sp_r * N_EDGES * DT / tau_rt
    ratio    = naive / correct if correct > 0 else float('inf')
    print(f'  {sp_r*1e9:>10.3f}  {delta_lam*1e9:>10.3f}  '
          f'{correct*1e9:>14.4f}  {naive*1e9:>12.2f}  {ratio:>8.1f}×')

## 6. Practical Implications

| Observation | Explanation |
|---|---|
| Resonance at $\lambda_c$ has **zero** shift | $\delta\lambda = 0$ → formula gives zero |
| Loss does **not** affect the shift | Loss only enters the amplitude condition; it has no effect on the resonant phase |
| Larger $\Delta t$ → larger shift | Linear in $n\,\Delta t / \tau_\text{rt}$ |
| Delay compensation of $k$ steps reduces $n_\text{eff}$ | Each compensated edge subtracts 1 from $n$ |
| Best carrier placement | Set $\lambda_c$ at the resonance of interest to minimise its shift |

**Key takeaway:** the artificial-delay shift in a baseband simulation is not
a fixed fraction of the FSR — it depends on how far each resonance sits from
$\lambda_c$.  To minimise the shift for a particular resonance, tune $\lambda_c$
to that resonance wavelength.